In [7]:
import pandas as pd
import os

# Preprocess Datasets into Train & Test

In [8]:
# load datasets
expr_train = pd.read_csv("../datasets/csv_files/deg_expression_matrix_train.csv")
clinical_train = pd.read_csv("../datasets/csv_files/clinical_metadata_train.csv")
expr_test_one = pd.read_csv("../datasets/csv_files/expression_matrix_test_one.csv")
clinical_test_one = pd.read_csv("../datasets/csv_files/clinical_metadata_test_one.csv")
expr_test_two = pd.read_csv("../datasets/csv_files/expression_matrix_test_two.csv")
clinical_test_two = pd.read_csv("../datasets/csv_files/clinical_metadata_test_two.csv")
expr_test_three = pd.read_csv("../datasets/csv_files/expression_matrix_test_three.csv")
clinical_test_three = pd.read_csv("../datasets/csv_files/clinical_metadata_test_three.csv")

In [9]:
# drop non-tumor samples from datasets
def filter_is_tumor(expr, clinical):
    samples_to_keep = list(clinical[clinical["is_tumor"] == 1]['sample_name'])
    expr_filtered = expr[expr['sample_name'].isin(samples_to_keep)]
    clinical_filtered = clinical[clinical["is_tumor"] == 1]
    return expr_filtered, clinical_filtered

In [10]:
def transpose_matrix(expr):
    expr_t = expr.set_index('gene_symbol').T
    expr_t.index.name = 'sample_name'
    expr_t.reset_index(inplace=True)
    expr_t.columns.name = None
    return expr_t

In [11]:
# add relapse free survival time & event to data
def construct_model_dataset(expr, clinical):
    if expr.columns[0] != "sample_name":
        expr = expr.drop(columns=expr.columns[0])

    gene_cols = [col for col in expr.columns if col != "sample_name"]

    expr = expr.reset_index(drop=True)
    clinical = clinical.reset_index(drop=True)

    data = pd.concat([
            expr["sample_name"],
            clinical[["relapse_free_time", "relapse_free_event"]],
            expr[gene_cols]
        ], axis=1)

    # drop null RFS values & rename columns
    data.dropna(inplace=True)
    data[["relapse_free_time", "relapse_free_event"]] = data[["relapse_free_time", "relapse_free_event"]].astype(int)

    data.head()
    return data

## Training

In [12]:
train_data = construct_model_dataset(*filter_is_tumor(expr_train, clinical_train))
train_data.head()

,sample_name,relapse_free_time,relapse_free_event,ACO1,ADAR,ADIPOQ,AGPAT2,ALDOC,ANG,ANGPT1,...,UBE2S,VAMP3,VAMP8,VAV3,VIM,VSIG4,VTCN1,WASF3,XG,ZBTB16
0,GSM1045208,3026,0,4.799942,11.395017,6.075966,6.143011,5.383726,5.904613,3.628884,...,4.844556,6.401260,9.204786,5.647802,6.964493,4.966453,7.220646,5.180080,3.151567,3.705057
1,GSM1045209,755,1,4.971609,10.864474,8.177897,7.368229,5.854996,6.582208,3.702770,...,6.627002,6.492745,8.080794,4.995820,7.585698,5.970720,5.437708,4.532965,3.383118,4.551614
2,GSM1045210,3014,0,4.555494,10.942121,6.828506,6.252349,5.317867,5.791332,3.630954,...,4.907463,6.269060,8.533085,5.194826,6.922194,5.289430,6.854089,4.636144,3.299654,4.031339
3,GSM1045211,406,1,4.668031,10.776268,5.196359,6.861987,6.408760,4.921507,3.693400,...,5.012260,6.170261,6.865168,5.039221,7.565630,5.307223,5.566723,4.642180,3.480787,4.412832
4,GSM1045212,2225,0,5.458046,11.743277,3.567575,6.902541,8.048325,5.603227,3.580604,...,7.380517,7.418157,9.662955,5.193608,8.330933,8.333481,10.065004,5.678931,3.197258,3.402715


## Testing

In [13]:
test_data_one = construct_model_dataset(*filter_is_tumor(transpose_matrix(expr_test_one), clinical_test_one))
test_data_one.head()

,sample_name,relapse_free_time,relapse_free_event,A1BG,A1BG-AS1,A1CF,A2M,A2M-AS1,A2ML1,A2MP1,...,ZWINT,ZXDA,ZXDB,ZXDC,ZYG11A,ZYG11B,ZYX,ZZEF1,ZZZ3,mir-223
0,GSM540108,731,1,6.146570,5.192952,4.136334,7.202705,5.378281,4.452781,4.456638,...,7.207004,4.956326,4.257066,7.258950,7.369622,6.635957,8.039407,6.266267,6.821724,4.431420
3,GSM540111,1770,1,5.717083,4.476920,3.887450,8.021876,5.039355,3.829404,3.530977,...,8.964749,5.658448,5.376926,6.416896,6.638156,7.852998,9.684551,5.897437,7.499180,4.209039
4,GSM540112,871,1,5.667256,5.047257,3.876345,7.659437,4.568131,3.678200,4.226887,...,9.051191,5.646992,5.464178,6.537697,5.560518,7.861657,8.696353,5.889624,7.680644,3.751116
5,GSM540113,301,1,5.821602,5.420414,3.737222,8.497944,5.923578,3.882873,3.602626,...,7.826577,5.197774,4.890690,6.444662,4.587187,8.174834,10.332553,6.433840,6.908704,4.178111
6,GSM540114,226,1,5.582540,4.874277,3.619344,8.218462,5.995361,3.741898,3.622146,...,9.342628,5.664366,5.156260,6.669061,5.831567,7.816731,9.150694,5.927160,7.267744,4.591735


In [14]:
test_data_two = construct_model_dataset(*filter_is_tumor(transpose_matrix(expr_test_two), clinical_test_two))
test_data_two.head()

,sample_name,relapse_free_time,relapse_free_event,A1BG,A1BG-AS1,A1CF,A2M,A2M-AS1,A2ML1,A2MP1,...,ZWINT,ZXDA,ZXDB,ZXDC,ZYG11A,ZYG11B,ZYX,ZZEF1,ZZZ3,mir-223
0,GSM519722,339,1,6.119304,4.600823,3.121685,7.234631,4.909697,2.937211,3.343626,...,9.184190,5.772036,4.551899,5.500941,4.855804,7.183321,7.117127,5.326840,6.888598,3.348883
1,GSM519723,445,1,6.479491,4.603668,3.268132,7.518227,5.186144,2.967149,3.629525,...,9.311570,5.413762,4.509711,5.397832,6.428620,7.418680,6.539444,5.408874,7.198831,3.170089
2,GSM519724,3155,0,6.017579,5.055174,3.115821,7.381913,7.623446,3.199144,3.517271,...,8.857554,6.072067,4.427425,5.508638,6.834819,7.409676,7.749218,5.416818,7.541631,4.304604
3,GSM519725,3068,0,6.061706,4.647037,3.169886,7.575412,6.109816,2.913530,3.463855,...,9.011089,5.943628,4.731611,5.801977,5.252555,7.610468,6.419296,5.246277,7.542585,3.766448
4,GSM519726,2633,0,5.581822,4.182169,3.329781,8.363513,6.142342,3.081503,3.650660,...,8.542251,5.563496,4.625013,5.748296,3.813661,7.285693,7.204199,5.308182,7.823545,3.683761


In [15]:
test_data_three = construct_model_dataset(*filter_is_tumor(transpose_matrix(expr_test_three), clinical_test_three))
test_data_three.head()

,sample_name,relapse_free_time,relapse_free_event,A1BG,A1BG-AS1,A1CF,A2M,A2M-AS1,A2ML1,A2MP1,...,ZWINT,ZXDA,ZXDB,ZXDC,ZYG11A,ZYG11B,ZYX,ZZEF1,ZZZ3,mir-223
0,GSM2345257,3981,1,6.823343,5.380914,3.815948,7.514131,5.307615,3.252120,3.979499,...,8.477204,6.867829,5.940665,6.698540,4.590683,8.376545,7.512789,6.010087,8.537435,4.594468
1,GSM2345258,3542,1,6.841661,4.960461,4.142036,8.185255,6.067712,3.498092,4.288013,...,8.445599,6.921858,5.754253,6.257776,5.806796,7.978497,7.391374,6.022865,8.616369,4.126529
2,GSM2345259,4090,0,7.517482,5.632821,4.079639,8.142596,6.313713,3.607268,4.457191,...,9.797934,6.916323,5.965644,6.457976,4.604947,7.660825,8.199574,5.975218,8.651620,4.773600
3,GSM2345260,2666,0,7.066259,5.161617,3.802018,8.159722,6.816215,3.310913,4.013709,...,8.284030,6.849369,5.764120,6.533928,4.688202,8.260489,7.425296,5.859670,8.214306,5.157202
4,GSM2345261,2885,1,7.407212,5.438206,3.970475,8.364816,6.534387,3.254299,4.338429,...,8.404814,7.343030,6.594634,6.685337,6.073278,8.370252,7.387143,5.847876,8.298858,4.396176


## Save Datasets

In [16]:
os.makedirs('../datasets/csv_files/ml_datasets', exist_ok=True)

train_data.to_csv('../datasets/csv_files/ml_datasets/train_data.csv', index=False)
test_data_one.to_csv('../datasets/csv_files/ml_datasets/test_data_one.csv', index=False)
test_data_two.to_csv('../datasets/csv_files/ml_datasets/test_data_two.csv', index=False)
test_data_three.to_csv('../datasets/csv_files/ml_datasets/test_data_three.csv', index=False)